In [1]:
import pandas as pd
import numpy as np
from BK_Tree import BKTree
import re
import string
from dead_processing_utils import *
from file_processing_utils import *

In [2]:
authority_hierarchy = strip('ucbg_templates/ucbg_4-0-0_authorityhierarchy-template.csv')
key = pd.read_csv('output/to_add_final.csv', dtype=str)
key['checked_synonym'] = key['checked_synonym'].apply(lambda x: str(x).split()[0])

Missing_hierarchies = pd.read_csv('output/taxon_hierarchy_check_nov2025.csv', dtype=str)
Missing_hierarchies = Missing_hierarchies[Missing_hierarchies['Has_Broader'] == 'no'].reset_index(drop=True)

In [3]:
def get_broader_from_sp_or_infra(name):
    if len(name.split()) == 2 or (len(name.split()) == 3 and 'Ã—' in name.split()):
        return name.split()[0]
    elif len(name.split()) > 2:
        return ' '.join(name.split()[:2])
    elif name in key['checked_synonym'].values:
        return key.loc[key['checked_synonym'] == name, 'family'].values[0].upper()

In [4]:
Missing_hierarchies['Broader_Term'] = (
    Missing_hierarchies['Name']
    .apply(get_broader_from_sp_or_infra))

In [5]:
Missing_hierarchies.head(50)

,Name,Has_Broader,Broader_Term
0,Acampe,no,ORCHIDACEAE
1,Acanthophippium,no,ORCHIDACEAE
2,Acanthosicyos,no,CUCURBITACEAE
3,Acianthus,no,ORCHIDACEAE
4,Acicarpha,no,CALYCERACEAE
5,Acmena,no,MYRTACEAE
6,Acostaea,no,ORCHIDACEAE
7,Acridocarpus,no,MALPIGHIACEAE
8,Acrocomia,no,ARECACEAE
9,Acronychia,no,RUTACEAE


In [6]:
authority_hierarchy['narrower_term'] = Missing_hierarchies['Name']
authority_hierarchy['broader_term'] = Missing_hierarchies['Broader_Term']
authority_hierarchy['term_type'] = 'taxonomyauthority'
authority_hierarchy['term_subtype'] = 'taxon'

In [7]:
db_key = pd.read_csv('data/noAuthorTest.csv')

db_key.rename(columns={'0': 'with_author'}, inplace=True)
db_key['with_author'] = db_key['with_author'].str.strip()
db_key['no_author'] = db_key['no_author'].str.strip()
db_key = db_key[['no_author', 'with_author']]


authority_hierarchy = authority_hierarchy.merge(
    db_key,
    how='left',
    left_on='narrower_term',
    right_on='no_author'
)

authority_hierarchy['narrower_term'] = authority_hierarchy['with_author'].fillna(
    authority_hierarchy['narrower_term']
)

authority_hierarchy = authority_hierarchy.drop(columns=['no_author', 'with_author'])

authority_hierarchy.to_csv('output/authorityhierarchy_test_upload_ROUND3.csv', index=False)